### Step 1 — Setup & Imports

In [1]:
!pip install "numpy<2"

In [2]:
import sklearn
print(sklearn.__version__)

/home/codespace/anaconda3/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/home/codespace/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/codespace/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (


1.6.0


In [3]:
import pandas as pd
import pickle

In [4]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import root_mean_squared_error

### Step 2 — MLflow Tracking Setup
Connects MLflow to the http://localhost:5000 wher mlflow is running and creates an experiment called nyc-taxi-experiment.

In [5]:
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("nyc-taxi-experiment")

2026/04/23 22:57:00 INFO mlflow.tracking.fluent: Experiment with name 'nyc-taxi-experiment' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1776985020988, experiment_id='1', last_update_time=1776985020988, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}>

### Step 3 — Proper Train/Validation Split
Refactors into a read_dataframe() function. Loads January as training data and February as validation data — a realistic time-based split.

In [13]:
def read_dataframe(filename):
    #We are only using parquet file now as the format has changed in the website
    df = pd.read_parquet(filename)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]
    
    #Feature Engineering
    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']

    return df

Training and Validation:

In [14]:
df_train = read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-01.parquet')
df_val = read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-02.parquet')

In [15]:
len(df_train), len(df_val)

(46307, 44218)

### Step 4 — DictVectorizer Fit/Transform
Fits the vectorizer on training data only (fit_transform), then applies it to validation (transform only) — this is the correct pattern to prevent data leakage.

In [16]:
categorical = ['PU_DO', 'trip_type'] #'PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [17]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

### Step 5 — XGBoost 
Converts data to XGBoost's DMatrix format. Defines an objective function that trains XGBoost with given params, logs everything to MLflow, and returns the RMSE as the loss.

In [20]:
import xgboost as xgb
from pathlib import Path

In [21]:
models_folder = Path('models')
models_folder.mkdir(exist_ok=True)

In [28]:
with mlflow.start_run():
    train = xgb.DMatrix(X_train, label=y_train)
    valid = xgb.DMatrix(X_val, label=y_val)

    best_params = {
    'learning_rate' : 0.2528037473043481,
    'max_depth' : 30,
    'min_child_weight' : 5.1810101322810915,
    'objective' : 'reg:linear',
    'reg_alpha' : 0.03263457245161076,
    'reg_lambda': 0.04365512556253237,
    'seed' : 42
    }

    mlflow.log_params(best_params)

    booster = xgb.train(
        params=best_params,
        dtrain=train,
        num_boost_round=50,
        evals=[(valid, 'validation')],
        early_stopping_rounds=50
    )

    y_pred = booster.predict(valid)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

    with open("models/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)
    mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

    mlflow.xgboost.log_model(booster, name="models_mlflow")

/home/codespace/anaconda3/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [23:36:03] WARNING: /workspace/src/objective/regression_obj.cu:227: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)


[0]	validation-rmse:7.72962
[1]	validation-rmse:6.83594
[2]	validation-rmse:6.27037
[3]	validation-rmse:5.92462
[4]	validation-rmse:5.71686
[5]	validation-rmse:5.58357
[6]	validation-rmse:5.50261
[7]	validation-rmse:5.45388
[8]	validation-rmse:5.42474
[9]	validation-rmse:5.40042
[10]	validation-rmse:5.38489
[11]	validation-rmse:5.37253
[12]	validation-rmse:5.36491
[13]	validation-rmse:5.35920
[14]	validation-rmse:5.35472
[15]	validation-rmse:5.35131
[16]	validation-rmse:5.34898
[17]	validation-rmse:5.34723
[18]	validation-rmse:5.34535
[19]	validation-rmse:5.34302
[20]	validation-rmse:5.34216
[21]	validation-rmse:5.34063
[22]	validation-rmse:5.33655
[23]	validation-rmse:5.33592
[24]	validation-rmse:5.33265
[25]	validation-rmse:5.33214
[26]	validation-rmse:5.33145
[27]	validation-rmse:5.33057
[28]	validation-rmse:5.33045
[29]	validation-rmse:5.32958
[30]	validation-rmse:5.32692
[31]	validation-rmse:5.32626
[32]	validation-rmse:5.32625
[33]	validation-rmse:5.32327
[34]	validation-rmse:5.3

/home/codespace/anaconda3/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [23:36:09] WARNING: /workspace/src/c_api/c_api.cc:1374: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  warnings.warn(smsg, UserWarning)
2026/04/23 23:36:12 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpz3n6q1v2/model, flavor: xgboost). Fall back to return ['xgboost==2.1.4']. Set logging level to DEBUG to see the full traceback. 
2026/04/23 23:36:12 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run brawny-hawk-782 at: http://localhost:5000/#/experiments/1/runs/5a12f157eae649f8a21251e87e92840d
🧪 View experiment at: http://localhost:5000/#/experiments/1
